# STRIDE AI Threat Modeler — Treino YOLOv8 v2 (111 classes)

**Pré-requisitos:**
1. Ativar GPU: `Runtime → Change runtime type → T4 GPU`
2. Fazer upload do dataset zipado no Google Drive (veja célula abaixo)

**Fluxo:**
```
Drive (dataset.zip) → Colab → YOLOv8s (50 epochs, GPU) → best.pt → Drive
```

## 0. Verificar GPU

In [ ]:
!nvidia-smi

## 1. Instalar dependências

In [ ]:
!pip install ultralytics -q

## 2. Montar Google Drive

**Antes de rodar esta célula:** faça upload do arquivo `dataset_augmented.zip` no seu Google Drive.

Para gerar o zip localmente, rode no terminal:
```bash
cd ~/Downloads/src/dataset
zip -r dataset_augmented.zip dataset_augmented/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Ajuste o caminho se necessário
DRIVE_ZIP = '/content/drive/MyDrive/dataset_augmented.zip'
assert os.path.exists(DRIVE_ZIP), f'Arquivo não encontrado: {DRIVE_ZIP}'

## 3. Extrair dataset

In [ ]:
!unzip -q /content/drive/MyDrive/dataset_augmented.zip -d /content/
!ls /content/dataset_augmented/ | head -5
!find /content/dataset_augmented -name '*.xml' | wc -l

## 4. Converter Pascal VOC XML → YOLO e preparar splits

In [ ]:
import os, shutil, random, xml.etree.ElementTree as ET
from pathlib import Path

# ── 111 classes (mesma ordem do projeto) ──────────────────────────────────────
COMPONENT_CLASSES_V2 = sorted([
    'api', 'aws_amazon_api_gateway', 'aws_amazon_cloudfront', 'aws_amazon_cloudwatch',
    'aws_amazon_dynamodb', 'aws_amazon_ec2', 'aws_amazon_ec2_auto_scaling',
    'aws_amazon_elastic_container_service', 'aws_amazon_elastic_kubernetes_service',
    'aws_amazon_elasticache', 'aws_amazon_rds', 'aws_amazon_redshift',
    'aws_amazon_route_53', 'aws_amazon_simple_notification_service',
    'aws_amazon_simple_queue_service', 'aws_amazon_simple_storage_service',
    'aws_amazon_virtual_private_cloud', 'aws_application_load_balancer',
    'aws_aurora_amazon_rds_instance', 'aws_auto_scaling', 'aws_autoscaling',
    'aws_backup', 'aws_cloud', 'aws_cloud_trail', 'aws_cloudformation',
    'aws_cloudformation_template', 'aws_cloudfront', 'aws_cloudwatch',
    'aws_dynamodb_table', 'aws_ec2_instance', 'aws_ec2_instances',
    'aws_elastic_block_store_volume', 'aws_elastic_container_service_container_2',
    'aws_elastic_container_service_service', 'aws_elastic_load_balancing',
    'aws_elastic_load_balancing_application_load_balancer',
    'aws_elastic_load_balancing_network_load_balancer', 'aws_elasticache',
    'aws_elactic_file_system(nfs)_multi-az', 'aws_identity_and_access_management',
    'aws_identity_access_management_role', 'aws_key_management_service',
    'aws_lambda', 'aws_lambda_lambda_function', 'aws_private_subnet',
    'aws_public_subnet', 'aws_rds', 'aws_region', 'aws_route_53_hosted_zone',
    'aws_simple_email_service', 'aws_simple_notification_service_topic',
    'aws_simple_queue_service_queue', 'aws_simple_storage_service_bucket',
    'aws_simple_storage_service_bucket_with_objects',
    'aws_simple_storage_service_object', 'aws_simple_storage_service_s3_standard',
    'aws_virtual_private_cloud', 'aws_vpc_virtual_private_cloud_vpc', 'aws_waf',
    'azure_api_management_services', 'azure_app_services', 'azure_application_insights',
    'azure_container_instances', 'azure_cosmos_db', 'azure_data_factories',
    'azure_databricks', 'azure_devops', 'azure_event_hubs', 'azure_firewalls',
    'azure_function_apps', 'azure_key_vaults', 'azure_kubernetes_services',
    'azure_load_balancers', 'azure_logic_apps', 'azure_machine_learning',
    'azure_machine_learning_studio_workspaces', 'azure_monitor', 'azure_network_security_groups',
    'azure_openai', 'azure_resource_groups', 'azure_services', 'azure_sql',
    'azure_sql_database', 'azure_sql_managed_instance', 'azure_sql_server',
    'azure_storage_accounts', 'azure_synapse_analytics', 'azure_virtual_machine',
    'azure_virtual_networks', 'azure_vm_scale_sets', 'developer_portal',
    'gcp_bigquery', 'gcp_cloud_functions', 'gcp_cloud_load_balancing',
    'gcp_cloud_run', 'gcp_cloud_sql', 'gcp_cloud_storage', 'gcp_compute_engine',
    'gcp_google_kubernetes_engine', 'gcp_identity_and_access_management',
    'gcp_pubsub', 'gcp_vertex_ai', 'gcp_virtual_private_cloud',
    'logic_apps', 'microsoft_entra', 'resource_group', 'sass_services',
    'sei/sip', 'solr',
])
CLASS_TO_IDX = {c: i for i, c in enumerate(COMPONENT_CLASSES_V2)}

SOURCE = Path('/content/dataset_augmented')
DEST   = Path('/content/dataset_v2')

def to_yolo(class_id, xmin, ymin, xmax, ymax, img_w, img_h):
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"

xml_files = list(SOURCE.rglob('*.xml'))
random.seed(42)
random.shuffle(xml_files)
n = len(xml_files)
splits = {
    'train': xml_files[:int(n*0.70)],
    'val':   xml_files[int(n*0.70):int(n*0.85)],
    'test':  xml_files[int(n*0.85):],
}

skipped = 0
for split, files in splits.items():
    (DEST / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DEST / 'labels' / split).mkdir(parents=True, exist_ok=True)
    for xml_path in files:
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            img_w = int(root.findtext('size/width', 0))
            img_h = int(root.findtext('size/height', 0))
            if img_w == 0 or img_h == 0:
                skipped += 1; continue
            img_name = root.findtext('filename', xml_path.stem + '.png')
            img_src = xml_path.parent / img_name
            if not img_src.exists():
                img_src = xml_path.with_suffix('.png')
            if not img_src.exists():
                skipped += 1; continue
            lines = []
            for obj in root.findall('object'):
                name = obj.findtext('name', '').strip().lower().replace(' ', '_')
                if name not in CLASS_TO_IDX:
                    continue
                bb = obj.find('bndbox')
                xmin = float(bb.findtext('xmin'))
                ymin = float(bb.findtext('ymin'))
                xmax = float(bb.findtext('xmax'))
                ymax = float(bb.findtext('ymax'))
                lines.append(to_yolo(CLASS_TO_IDX[name], xmin, ymin, xmax, ymax, img_w, img_h))
            if not lines:
                skipped += 1; continue
            shutil.copy2(img_src, DEST / 'images' / split / img_src.name)
            (DEST / 'labels' / split / (xml_path.stem + '.txt')).write_text('\n'.join(lines))
        except Exception:
            skipped += 1

# data.yaml
yaml_content = f"""path: {DEST}
train: images/train
val: images/val
test: images/test
nc: {len(COMPONENT_CLASSES_V2)}
names: {COMPONENT_CLASSES_V2}
"""
(DEST / 'data.yaml').write_text(yaml_content)

total = sum(len(v) for v in splits.values())
print(f'Dataset pronto: {total - skipped} imagens válidas ({skipped} ignoradas)')
print(f'Train: {len(splits["train"])} | Val: {len(splits["val"])} | Test: {len(splits["test"])}')

## 5. Treinar YOLOv8s — 50 epochs (GPU)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # transfer learning

results = model.train(
    data='/content/dataset_v2/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,          # GPU T4 aguenta batch 32
    optimizer='AdamW',
    lr0=0.001,
    patience=15,
    project='/content/models',
    name='arch_detector_v2',
    exist_ok=True,
    augment=True,
    verbose=True,
)

print('\nTreino concluído!')
print(f'mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

## 6. Avaliar no conjunto de teste

In [ ]:
best = YOLO('/content/models/arch_detector_v2/weights/best.pt')
metrics = best.val(data='/content/dataset_v2/data.yaml', split='test')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 7. Salvar modelo no Google Drive

In [ ]:
import shutil
from pathlib import Path

src  = Path('/content/models/arch_detector_v2/weights/best.pt')
dest = Path('/content/drive/MyDrive/stride_model_v2_best.pt')
shutil.copy2(src, dest)
print(f'Modelo salvo em: {dest}')
print(f'Tamanho: {dest.stat().st_size / 1024 / 1024:.1f} MB')

## 8. Download direto (alternativa ao Drive)

Se preferir baixar direto para o computador em vez do Drive:

In [ ]:
from google.colab import files
files.download('/content/models/arch_detector_v2/weights/best.pt')